# 02 - Data Cleaning & Schema Generation

In this notebook, we walk through the data cleaning process interactively. We will:
1. Clean the `nav_history`, `investor_transactions`, and `scheme_performance` datasets.
2. Standardize dates, handle anomalies, and ensure data quality before they are loaded into our SQLite database.

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path

BASE_DIR = Path().resolve().parent
RAW_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'

## 1. Clean NAV History
Parse dates, remove missing/invalid NAVs, and forward-fill for weekends/holidays.

In [ ]:
df_nav = pd.read_csv(RAW_DIR / '02_nav_history.csv')
df_nav.drop_duplicates(inplace=True)

# Parse dates and filter valid NAV
df_nav['date'] = pd.to_datetime(df_nav['date'], format='mixed', dayfirst=True)
df_nav = df_nav[df_nav['nav'] > 0]
df_nav.set_index('date', inplace=True)

# Forward fill per fund
cleaned_dfs = []
for code, group in df_nav.groupby('amfi_code'):
    group = group[~group.index.duplicated(keep='first')]
    full_range = pd.date_range(start=group.index.min(), end=group.index.max(), freq='D')
    group_reindexed = group.reindex(full_range)
    group_reindexed['amfi_code'] = code
    group_reindexed['nav'] = group_reindexed['nav'].ffill()
    cleaned_dfs.append(group_reindexed)

df_nav_clean = pd.concat(cleaned_dfs).reset_index().rename(columns={'index': 'date'})
df_nav_clean['date'] = df_nav_clean['date'].dt.strftime('%Y-%m-%d')
print(f"Cleaned NAV shape: {df_nav_clean.shape}")
display(df_nav_clean.head())

## 2. Clean Investor Transactions
Standardize transaction types, validate amounts, and clean date formats.

In [ ]:
df_txn = pd.read_csv(RAW_DIR / '08_investor_transactions.csv')
df_txn['transaction_date'] = pd.to_datetime(df_txn['transaction_date'], format='mixed').dt.strftime('%Y-%m-%d')

df_txn['transaction_type'] = df_txn['transaction_type'].str.upper().str.strip()
valid_types = ['SIP', 'LUMPSUM', 'REDEMPTION']
df_txn.loc[~df_txn['transaction_type'].isin(valid_types), 'transaction_type'] = 'OTHER'

df_txn = df_txn[df_txn['amount_inr'] > 0]
df_txn['kyc_status'] = df_txn['kyc_status'].str.upper()

print(f"Cleaned Transactions shape: {df_txn.shape}")
display(df_txn.head())

## 3. Clean Scheme Performance
Ensure numeric types for returns and handle outlier expense ratios.

In [ ]:
df_perf = pd.read_csv(RAW_DIR / '07_scheme_performance.csv')

return_cols = ['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct']
for col in return_cols:
    df_perf[col] = pd.to_numeric(df_perf[col], errors='coerce')

# Clamp expense ratios to valid range (0.1% - 2.5%)
df_perf['expense_ratio_pct'] = df_perf['expense_ratio_pct'].clip(lower=0.1, upper=2.5)

print(f"Cleaned Scheme Performance shape: {df_perf.shape}")
display(df_perf.head())

## Final Note on SQLite DB Setup
The complete process of saving these processed files to `data/processed/` and executing `schema.sql` to generate the Star Schema database (`bluestock_mf.db`) is fully automated in the `scripts/02_data_cleaning.py` script. 

This notebook just serves as an interactive view of how the cleaning logic works!